# Quantum noise, the quantum way: trajectory (quantum-jump) simulation of the surface code

---

Notebooks 1–7 built, in stages, a matrix-product-state (MPS) simulator for the rotated planar
surface code — a software Pauli frame, an online sliding-window union-find decoder, transversal
Clifford gates, magic-state $S/T$ teleportation — and a **stochastic-Pauli** noise model validated at
$d=3$. Notebook 8 pushed the distance up; notebook 9 showed that **tsim** (a ZX stabiliser-rank
sampler with native $T$) does almost everything that engine does for stochastic-Pauli noise, *far*
faster.

So why keep the tensor-network engine at all? Because the two simulators make **opposite** things
cheap, and there is a large and physically important class of noise that lives entirely on the side
tsim cannot reach. This notebook is about **that** class — and about turning the engine from a
teaching tool into something that could, on an HPC cluster, be a genuine research instrument for
**validating QEC against realistic device noise**.

We do not run the heavy sweeps here (they are a cluster-scale job — see §9). The deliverable is the
**method, the code that implements it, and the argument for what it unlocks.**

## §1 — The core asymmetry: what is cheap where

| | **tsim** (stabiliser rank) | **ITensor MPS** (full Hilbert space) |
|---|:---:|:---:|
| the *state* | **cheap** — a sum of stabiliser states (linear in Clifford gates, $2^{\#T}$ in magic) | **expensive** — amplitudes tracked explicitly, bond dimension $\chi$ grows |
| the *noise* | **restricted** — must be stochastic **Pauli** to stay in the formalism | **free-form** — *any* CPTP channel is just more operators on the state |

> In the **exact** picture, noise generality is free and the state is the cost.
> In the **stabiliser** picture, the state is free and the noise is restricted to Pauli.

That restriction is not incidental — it is *what lets* tsim Monte-Carlo over Pauli frames and decode
from classical syndromes. Everything genuinely non-Pauli is off-limits to it: coherent
over-rotations, $T_1$ amplitude damping, $T_2$ non-Pauli dephasing, leakage out of the qubit
subspace, correlated/non-Markovian baths, and the exact logical *channel* (as opposed to sampled
bits). Every one of those is a real effect on real hardware, and every one is **native** to a
state-vector/MPS engine: you just apply the operator.

The vehicle that makes this tractable — keeping us on *pure states* instead of paying the $2^{2n}$
density-matrix cost — is the **quantum trajectory**.

## §2 — Load the engine

`trajectory_engine.jl` `include`s the full `../8.Scaling/scaling_engine.jl` (Code object, union-find
decoder, transversal gates, magic-state teleportation, frame bookkeeping, the exact non-collapsing
readout oracle) **unchanged**, and adds the quantum-noise layer on top. Nothing in the scaling engine
is modified — the trajectory machinery is purely additive.

In [ ]:
include("trajectory_engine.jl");

## §3 — Quantum trajectories in one paragraph

A completely-positive trace-preserving (CPTP) channel is written in **Kraus form**
$$\rho \;\longmapsto\; \sum_i K_i\,\rho\,K_i^\dagger, \qquad \sum_i K_i^\dagger K_i = \mathbb{I}.$$
Simulating this directly needs the density matrix $\rho$ — for $n$ qubits that is a $2^{n}\times 2^{n}$
object, i.e. $2^{2n}$ numbers, squaring the already-hard MPS cost.

The **quantum-trajectory** (quantum-jump / Monte-Carlo wavefunction) unravelling avoids that. On a
*pure* state $|\psi\rangle$ we realise the same channel stochastically: draw **one** Kraus branch $i$
with its Born probability
$$p_i \;=\; \langle\psi|\,K_i^\dagger K_i\,|\psi\rangle,$$
apply that $K_i$, and renormalise. The state stays pure. Any observable, **averaged over many
independent trajectories**, reproduces the exact open-system value:
$$\mathbb{E}\big[\langle\psi|\,O\,|\psi\rangle\big] \;=\; \operatorname{Tr}(\rho\,O).$$

So we pay $2^n$ (pure state) per run and Monte-Carlo the dissipation over runs, instead of $2^{2n}$
once. That is the *entire* trick — and it fits the existing pure-state MPS engine with **no**
structural change. The price is statistical: we need enough trajectories to beat down the sampling
error. Which is exactly the kind of workload a cluster eats for breakfast (§9).

## §4 — The channels: none of these is a Pauli mixture

Below are the single-qubit channels the engine ships, each as a Kraus set $\{K_i\}$. Amplitude and
phase damping are the canonical $T_1$/$T_2$ processes; **neither is a probabilistic mixture of
Paulis**, so both are outside tsim's efficient regime. (Depolarising *is* a Pauli mixture — included
as the bridge back to the notebook-7 model.) `iscptp` checks $\sum_i K_i^\dagger K_i = \mathbb{I}$.

In [ ]:
for (name, K) in (("amp_damp(0.05)  — T1 relaxation", amp_damp(0.05)),
                  ("phase_damp(0.05) — T2 dephasing", phase_damp(0.05)),
                  ("depolarize(0.05) — Pauli mixture", depolarize(0.05)),
                  ("compose(amp_damp(0.05), phase_damp(0.05)) — simultaneous T1+T2",
                                        compose(amp_damp(0.05), phase_damp(0.05))))
    println(rpad(name, 60), "  #Kraus=", length(K), "   CPTP: ", iscptp(K))
end

### The jump step

`apply_channel!(psi, s, kraus)` is the trajectory unravelling itself: score every branch by its Born
probability, sample one, apply it, renormalise. It routes every apply through the scaling engine's
`_ap`, so the bond-dimension diagnostics (`CHIMAX`, `CAP_HITS`) keep working. **It is
dimension-agnostic** — the same function will handle the qutrit leakage site in §6 with no change,
because all it does is multiply matrices and take inner products.

## §5 — The headline advantage: coherent errors

This is the single most important thing the exact engine can do that tsim cannot.

A systematic **over-rotation** $R_z(\varepsilon)$ (a miscalibrated phase gate, calibration drift) or
an always-on **$ZZ$ crosstalk** term is a *unitary* — a definite, coherent rotation, not a random
Pauli. tsim can only insert its **Pauli twirl**: the stochastic channel with the same *average*
fidelity. That approximation gets the average right and the **worst case wrong**:

* a coherent rotation by $\varepsilon$ has average infidelity $\sim\varepsilon^2$ but diamond-norm
  error $\sim\varepsilon$;
* coherent errors can add in **amplitude** across gates ($\sim(N\varepsilon)^2$) rather than
  random-walking ($\sim N\varepsilon^2$).

Whether a QEC cycle "twirls away" that coherence — the syndrome projection partially does — is a
genuine research question that is answerable **only** by tracking amplitudes. The exact engine can
insert the *real* rotation and, via the non-collapsing oracle, read out the exact logical channel to
**quantify** the gap. `twirl_Rz(ε)` returns precisely the dephasing channel tsim would substitute, so
the two can be compared at matched average fidelity.

In [ ]:
ε = 0.15
println("coherent  Rz(ε):                 ", round.(Rz(ε), digits=4))
println("its Pauli twirl (what tsim uses): Z with prob sin²(ε/2) = ", round(sin(ε/2)^2, digits=5))
println("twirl_Rz(ε) Kraus set CPTP:       ", iscptp(twirl_Rz(ε)))
# A ChannelModel carrying the REAL coherent rotation (no twirl) on every data qubit each round:
cm_coherent = ChannelModel(; coh = Rz(ε))
# ...vs the stochastic-Pauli channel tsim is forced to approximate it by:
cm_twirled  = ChannelModel(; kraus1 = twirl_Rz(ε))
println("\nBuilt two matched-fidelity models: cm_coherent (exact unitary) and cm_twirled (Pauli).")

**The experiment this enables** (a cluster job, sketched — not run here): take a magic-Bell
circuit, expose it to `cm_coherent` and to `cm_twirled`, and compare the *exact* logical correlators
via `mean_correlator_traj`. Under the coherent model the off-diagonal $\langle XX\rangle$ drifts
*coherently* (a rotation of the logical state); under the twirled model it simply *decays*. Matched
average fidelity, different logical channel — the quantitative statement of what the Pauli
approximation discards. This is future-work item 1 ("coherent over-rotation study") and item 2
("Pauli-twirl-approximation validation") from the roadmap, now with the code to run them:

In [ ]:
# ILLUSTRATIVE — a single coherent-vs-twirled comparison (each call is many trajectories; run on a cluster).
# c   = build_code(3)
# circ = [(:H,1),(:T,1),(:CNOT,1,2)]            # magic Bell; exact clean ⟨XX⟩ = cos(π/4) ≈ 0.707
# set_precision!(cutoff=1e-6, maxdim=64)
# xx_coh,  se_coh  = mean_correlator_traj(c, circ, :X, :X, cm_coherent, 2000; R=4, seed=1)
# xx_twirl,se_twirl= mean_correlator_traj(c, circ, :X, :X, cm_twirled,  2000; R=4, seed=1)
# @printf("⟨XX⟩  coherent = %.4f ± %.4f    twirled = %.4f ± %.4f    gap = %.4f\n",
#         xx_coh, se_coh, xx_twirl, se_twirl, xx_coh - xx_twirl)
println("(left commented: this is the cluster-scale measurement, not a live cell — see §9)")

## §6 — Putting it in the QEC loop

`run_epoch_traj!` runs one idle epoch as a single trajectory. Each round, in order: coherent
unitary → $ZZ$ crosstalk → the stochastic single-qubit channel (a live quantum jump) → a raw syndrome
round on the **corrupted** MPS → classical measurement flips. Because we corrupt the *real* state,
the measured syndromes automatically reflect the channel — there is no outcome-hacking. The epoch
then sliding-decodes into the software frame exactly as the noiseless engine does.

`estimate_pL_traj` is the trajectory analogue of the scaling engine's `estimate_pL`: run $N$
trajectories, count those whose frame-corrected $\pm1$ oracle comes back negative, return the failure
fraction and its binomial error. **Same control flow — only the noise is now a genuine CPTP
channel.** Here is the exact drop-in relationship:

In [ ]:
# The notebook-7 / scaling-engine estimator (stochastic PAULI):
#     estimate_pL(c, :zero, :Z, R, NoiseModel(; p=..., q=...), N)
# The trajectory estimator (arbitrary QUANTUM channel) — identical shape:
#     estimate_pL_traj(c, :zero, :Z, R, ChannelModel(; kraus1=amp_damp(...), q=...), N)

# Example model definitions (T1 + coherent drift + crosstalk + readout flips) — ready to sweep:
cm_T1        = ChannelModel(; kraus1 = amp_damp(0.01), q = 0.01)
cm_T1T2      = ChannelModel(; kraus1 = compose(amp_damp(0.01), phase_damp(0.01)), q = 0.01)
cm_realistic = ChannelModel(; kraus1 = compose(amp_damp(0.008), phase_damp(0.004)),
                              coh = Rz(0.05), zz = 0.01, q = 0.008)
println("Three physical channel models built (T1; T1+T2; T1+T2+coherent-drift+crosstalk+readout).")
println("Each is a drop-in for estimate_pL_traj — see the commented cluster call below.")

In [ ]:
# ILLUSTRATIVE — a p_L point under genuine T1 noise (many trajectories → run on a cluster):
# c = build_code(3); set_precision!(cutoff=1e-6, maxdim=64)
# pL, se = estimate_pL_traj(c, :zero, :Z, 4, cm_T1, 4000; C=2, B=2, seed=100)
# @printf("d=3 Z-memory under amplitude damping:  pL = %.4f ± %.4f  (N=4000, R=4)\n", pL, se)
println("(left commented — cluster-scale; the code path is exercised, the statistics are the cost)")

## §7 — Leakage: the trajectory step is dimension-agnostic

Leakage is population escaping the qubit subspace $\{|0\rangle,|1\rangle\}$ into a third level
$|2\rangle$ — a transmon's second excited state, a neutral-atom Rydberg/hyperfine level (this
project's architecture). It has three properties that make it harder than any Pauli: it is **off-subspace** (no Pauli
describes a non-qubit), **sticky** (lifetimes span many rounds → a *time-extended streak* of
detection events), and **spreading** (a leaked qubit corrupts every gate it touches). The stabiliser
formalism is fundamentally 2-level; it cannot represent $|2\rangle$ **at all**.

The exact engine can — you raise the site dimension to 3. And here is the payoff of doing it with
trajectories rather than a density matrix: the honest leakage density matrix costs $3^{2n}$; a
**trajectory keeps it at $3^{n}$ amplitudes**. That is what turns "definable" into "feasible."

The remarkable part: **`apply_channel!` needs no change.** It only multiplies matrices and takes
inner products, so a $3\times3$ Kraus set on a `dim=3` site works byte-for-byte. `leak_channel(γL, γS)`
is the leakage/seepage channel; below we confirm it is trace-preserving on the qutrit and run one
live jump on a genuine 3-level ITensor site — the *same* jump machinery the qubits use.

In [ ]:
γL, γS = 0.02, 0.30                       # leak prob per round, seep-back prob per round
K = leak_channel(γL, γS)
println("leak_channel(γL,γS) has ", length(K), " Kraus ops; CPTP on the qutrit: ", iscptp(K))

# One live quantum jump on a real dim-3 ITensor site — apply_channel! unchanged from the qubit case.
s3  = qudit_site(3)
ψ1  = MPS([ITensor(ComplexF64[0, 1, 0], s3)])   # start in |1⟩ (basis index 2 of the 3-level site)
Random.seed!(7)
ψ1, branch = apply_channel!(ψ1, s3, K)
println("started in |1⟩; sampled Kraus branch = ", branch, "  (1=no-jump, 2=leak→|2⟩, 3=seep)")
# population that has leaked into |2⟩ after the jump (per this trajectory):
n2 = op([0 0 0; 0 0 0; 0 0 1], s3)
println("⟨n₂⟩ on this trajectory = ", round(real(inner(ψ1', apply(n2, ψ1))), digits=3))

A *full* leakage engine (Tier C of the design doc) also raises the **gate and measurement**
layer to three levels — every `CNOT`, `H`, projector, and reset needs a defined action on $|2\rangle$,
plus leakage-reduction units (LRUs) and heralding/erasure conversion. That is a from-scratch operator
layer, deliberately out of scope here. But the genuinely *new physics* — the off-subspace channel and
its Monte-Carlo unravelling — is exactly what we have just shown runs on the existing trajectory step.
The trajectory approach is what makes carrying a qutrit per site affordable enough to be worth
building the rest.

## §8 — Non-Markovian noise: NMQSD / HOPS on a tensor network

Everything above assumes **Markovian** noise: each round's channel is memoryless, independent of the
bath's past. Real devices are not so kind — $1/f$ charge noise, two-level-system (TLS) defects,
non-flat spectral densities, and slow drifts all give the environment a **memory**. The bath
"remembers" what the qubit did and feeds it back. No per-round Kraus map can capture that, because the
map itself would have to depend on history.

The pure-state way to simulate genuinely non-Markovian open dynamics is **non-Markovian quantum state
diffusion (NMQSD)** and its close relative the **Hierarchy Of Pure States (HOPS)**. The system obeys a
stochastic Schrödinger equation driven by a **coloured** noise process $z_t$ whose correlation
function
$$\langle z_t\, z_s^{*}\rangle \;=\; \alpha(t-s) \;=\; \tfrac{1}{\pi}\!\int_0^\infty\! d\omega\; J(\omega)\,
\big[\coth(\beta\omega/2)\cos\omega\tau - i\sin\omega\tau\big]$$
set by the bath **spectral density** $J(\omega)$ — i.e. the memory kernel is the physical bath, not a
white-noise idealisation. HOPS lifts that non-local-in-time memory into a **hierarchy of auxiliary
pure states** indexed by a multi-index $\mathbf{k}$; each auxiliary state obeys a *local*-in-time
equation coupled to its neighbours in the hierarchy, and truncating the hierarchy depth truncates the
memory. Every member of the hierarchy is a pure state — so it Monte-Carlos over $z_t$ exactly like the
quantum jumps above, just with a structured, correlated noise instead of white.

**Why tensor networks, and why this engine.** The auxiliary states are themselves many-body wave
functions, and the hierarchy index is *another tensor leg*: the whole object is a tensor network, so
MPS/ITensor compresses it — this is precisely the method of
[arXiv:2108.06224](https://arxiv.org/abs/2108.06224) (*Many-body quantum state diffusion for
non-Markovian dynamics in strongly interacting systems*, Flannigan, Damanet & Daley), which combines
NMQSD with tensor-network methods for a dissipative many-body model. Dropping a surface-code patch in
as the "system" and a device-calibrated $J(\omega)$ (e.g. a Lorentzian TLS peak, or measured $1/f$)
as the bath is a direct application. **A stabiliser simulator cannot go anywhere near this:** the
system–bath state is coherently entangled and non-Pauli by construction; there is no Clifford frame to
sample. This is the deepest structural advantage of the tensor-network engine — it is not "a slower
Pauli sampler," it is a different and strictly larger class of physics.

*Scope note:* wiring a full HOPS hierarchy onto the surface-code MPS is a substantial build (its own
project), not a cell in this notebook. It is included here because it is the natural end-point of the
"noise the quantum way" story and the clearest example of something only a tensor-network engine can
do — and because the method already exists in the group's own work.

## §9 — Scaling this on an HPC cluster

The trajectory method is **embarrassingly parallel**, in three independent directions at once:

1. **Over trajectories.** Each trajectory is an independent run with its own random seed; they never
   communicate. $N$ trajectories on $M$ cores is a near-perfect linear speedup, and the only thing
   that comes back from each worker is a handful of scalars (a $\pm1$ oracle bit, or a correlator
   value). Statistical error falls as $1/\sqrt{N}$, so a cluster buys precision directly.
2. **Over the physics sweep.** Noise strength $p$, the channel *type* (T1 vs T2 vs coherent vs
   leakage vs correlated), distance $d$, readout basis, T-count — every axis is another independent
   batch of jobs.
3. **Within a trajectory** (secondary): the MPS applies and the SVD can use multithreaded BLAS.

Concretely: one $d=3$ memory trajectory is milliseconds-to-seconds of MPS work; a well-resolved $p_L$
point ($N\sim10^4$) is a few core-hours; a full channel-comparison campaign across a $p$-sweep, several
channels, and $d\in\{3,5\}$ is $10^3$–$10^5$ core-hours — a routine cluster allocation, and each job
is a one-line `estimate_pL_traj` / `mean_correlator_traj` call with a different seed and model. The
driver is trivial; here is the shape of it with Julia's `Distributed`:

In [ ]:
# ILLUSTRATIVE driver — the parallel structure, not a live run.
# using Distributed
# addprocs(64)                                   # or launch one Julia process per cluster node
# @everywhere include("trajectory_engine.jl")
# @everywhere const C3 = build_code(3)
#
# ps       = [0.001, 0.002, 0.005, 0.01, 0.02]   # noise-strength sweep
# channels = Dict("T1"      => p -> ChannelModel(; kraus1 = amp_damp(p),  q = p),
#                 "T2"      => p -> ChannelModel(; kraus1 = phase_damp(p),q = p),
#                 "coh"     => p -> ChannelModel(; coh = Rz(sqrt(p)),     q = p),
#                 "depol"   => p -> ChannelModel(; kraus1 = depolarize(p),q = p))
# jobs = [(name, p) for name in keys(channels) for p in ps]        # every (channel, p) is independent
#
# results = pmap(jobs) do (name, p)               # pmap distributes jobs across all workers
#     set_precision!(cutoff = 1e-6, maxdim = 64)
#     pL, se = estimate_pL_traj(C3, :zero, :Z, 4, channels[name](p), 10_000; seed = hash((name, p)))
#     (name, p, pL, se)
# end
# # gather → one logical-error-vs-p curve PER channel: exactly the plot that validates a device.
println("(commented: the cluster driver — trajectories/sweeps fan out with a single pmap)")

## §10 — What else the tensor engine can do that tsim cannot (and the honest limits)

Collecting the advantages, with the cheapest wins first:

| Capability | tsim (stabiliser rank) | ITensor MPS + trajectories |
|---|:---:|:---:|
| Stochastic Pauli noise | ✅ native, **fast** | ✅ (slower) |
| Coherent / unitary errors, **stochastic** | ❌ (Pauli-twirl only) | ✅ real unitary (§5) |
| Amplitude damping $T_1$ / phase damping $T_2$ | ❌ | ✅ (§4, §6) |
| General non-Pauli CPTP / Lindblad channels | ❌ | ✅ (any Kraus set) |
| **Leakage** (out of the qubit subspace) | ❌ (2-level formalism) | ✅ raise site dim (§7) |
| **Correlated / non-Markovian** noise (structured bath) | ❌ | ✅ NMQSD/HOPS (§8) |
| Always-on $ZZ$ crosstalk, calibration drift | ❌ (or $2^{\#}$ as fixed gates) | ✅ (§5, §6) |
| Exact non-collapsing observables / **logical channel** | ❌ (samples bits) | ✅ (`corr2`, `mean_correlator_traj`) |
| Pauli-twirl-approximation **validation** | ❌ (it *is* the approximation) | ✅ (§5) |
| Large distance $d$, high shot count, thresholds | ✅ **owns this** | ❌ (state cost) |

Two more that fall out for free once the state is exact: **continuous weak measurement / measurement
back-action** (a partial projector is just another Kraus operator), and **non-Pauli readout errors**
(a soft/analog measurement channel rather than a classical bit-flip).

## §11 — Bottom line: from teaching tool to research instrument

The stochastic-Pauli engine of notebooks 7–9 is a faithful, tsim-checkable teaching model. This
notebook shows that the *same* MPS engine, with a small additive layer, becomes something tsim
**structurally cannot be**: a carrier of arbitrary quantum noise that reports exact logical
quantities.

* **Quantum trajectories** unravel any CPTP channel on the pure-state MPS ($2^n$, not $2^{2n}$),
  averaging to the exact open-system result and fanning out perfectly across a cluster.
* The **headline win — coherent errors — is cheap and pure-state**: stop twirling, insert the real
  $R_z(\varepsilon)$ / $ZZ$ unitary, and read the exact logical channel to *measure* what the Pauli
  twirl discards.
* **$T_1$/$T_2$ and other non-Pauli channels** slot in as Kraus sets with the identical estimator.
* **Leakage** needs only a bigger local Hilbert space, and the trajectory step handles the qutrit
  **unchanged** — trajectories are what make it affordable.
* **Non-Markovian, correlated baths** are reachable via **NMQSD/HOPS on a tensor network**
  ([arXiv:2108.06224](https://arxiv.org/abs/2108.06224)) — a class of physics with no stabiliser
  analogue at all.

The price is system size: this is the **ground-truth companion** to tsim's scalable sampling, not a
replacement. tsim owns large-$d$ thresholds; the tensor engine owns *realism* — validating QEC
performance against the actual physical noise processes of a device (coherent, leaky, dissipative,
correlated) at small $d$, as the reference every faster, more approximate simulator is ultimately
checked against.